In [1]:
import yaml
import argparse
import logging
from types import SimpleNamespace
import os
from pathlib import Path
import csv
import sys
import math
import numpy as np
import pandas as pd
import time
import joblib
import json
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
sys.path.append('D:\\PythonPrograms\\GraspDataProcessing\\src')
import graspdataprocessing as gdp

def load_config(config_path):
    """加载YAML配置文件"""
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    return SimpleNamespace(**config)

def update_config(config_path, updates):
    """更新YAML配置文件
    
    Args:
        config_path: 配置文件路径
        updates: 要更新的键值对字典
    """
    with open(config_path, 'r') as f:
        config = yaml.safe_load(f)
    
    # 更新配置值
    config.update(updates)
    
    with open(config_path, 'w') as f:
        yaml.dump(config, f, default_flow_style=False, sort_keys=False)


def setup_logging(config):
    """配置日志系统"""
    os.makedirs("logs", exist_ok=True)
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(levelname)s - %(message)s',
        handlers=[
            logging.FileHandler("logs/machine_learning_training.log"),
            logging.StreamHandler()
        ]
    )
    return logging.getLogger(__name__)

def traning_model(config):
    """训练模型"""
    logger = setup_logging(config)
    logger.info("开始训练模型")

In [2]:
config_file = 'config.yaml'  # 配置文件的路径
config = load_config(config_file)

In [4]:

config.file_name = f'{config.conf}_{config.cal_loop_num}'
"""主程序逻辑"""
logger = setup_logging(config)
logger.info("机器学习训练程序启动")
excution_time = time.time()

# 使用 pathlib 创建目录
root_path = Path(config.root_path)
cal_path = root_path.joinpath(f'{config.conf}_{config.cal_loop_num}')
(root_path / "models").mkdir(parents=True, exist_ok=True)
(root_path / "descripotors").mkdir(parents=True, exist_ok=True)
(root_path / "descripotors_stay").mkdir(parents=True, exist_ok=True)
(root_path / "test_data").mkdir(parents=True, exist_ok=True)
(root_path / "roc_curves").mkdir(parents=True, exist_ok=True)
(root_path / "results").mkdir(parents=True, exist_ok=True)

logger.info(f"初始比例: {config.initial_ratio}")
logger.info(f"光谱项: {config.spetral_term}")

# 使用 pathlib 构建路径并创建CSV文件
result_csv_path = root_path.joinpath('results/results.csv')
try:
    if not result_csv_path.exists():
        with result_csv_path.open(mode="w", newline="") as file:
            writer = csv.writer(file)
            writer.writerow([
                'training_time', 'eval_time', 'abinitio_time', 'all_time',
                'f1', 'roc_auc', 'accuracy', 'precision', 'recall',
                'Es', 'abimport_csfnum', 'MLimport_csfnum', 'MLsampling_ratio', 'next_itr_num',
                'weight', 'f1_train', 'roc_auc_train', 'accuracy_train', 'precision_train', 'recall_train'
            ])
except IOError as e:
    logger.error(f"无法创建结果文件 {result_csv_path}: {str(e)}")
    raise

2025-06-03 23:58:48,599 - INFO - 机器学习训练程序启动
2025-06-03 23:58:48,601 - INFO - 初始比例: 0.09
2025-06-03 23:58:48,601 - INFO - 光谱项: ['5s(2).4d(10)1S0_1S.5p(6).6s(2).4f(7)8S0_8S.5d_9D', '5s(2).4d(10)1S0_1S.5p(6).6s(2).4f(7)8S0_8S.5d_7D']


In [4]:

initial_csfs_path = root_path.joinpath(config.target_pool_file)
try:
    if not initial_csfs_path.is_file():  # 检查是否是有效文件
        logger.error(f"初始CSFs文件无效或不存在: {initial_csfs_path}")
        raise FileNotFoundError(f"初始CSFs文件无效或不存在: {initial_csfs_path}")
    logger.info(f"成功加载初始CSFs文件: {initial_csfs_path}")
except PermissionError as e:
    logger.error(f"无权限访问CSFs文件: {initial_csfs_path}")
    raise
except Exception as e:
    logger.error(f"加载CSFs文件时发生未知错误: {str(e)}")
    raise

2025-06-02 21:46:56,193 - INFO - 成功加载初始CSFs文件: /home/workstation3/caldata/GdI/cvodd1/as3_odd4/cv4odd1as3_odd4.c


In [5]:
energy_level_file_path = cal_path.joinpath(f'{config.conf}_{config.cal_loop_num}.level')
energy_level_file_load = gdp.LevelsEnergyData.from_filepath(energy_level_file_path)

energy_level_data_pd = energy_level_file_load.energy_level_2_pd()

cal_configuration_set = set(energy_level_data_pd['configuration'])

Data file /home/workstation3/caldata/GdI/cvodd1/as3_odd4/cv4odd1as3_odd4_1/cv4odd1as3_odd4_1.level loaded.
file /home/workstation3/caldata/GdI/cvodd1/as3_odd4/cv4odd1as3_odd4_1/cv4odd1as3_odd4_1.level loaded
data file type: level data
energy levels data has been written into /home/workstation3/caldata/GdI/cvodd1/as3_odd4/cv4odd1as3_odd4_1/level_csv_file/cv4odd1as3_odd4_1.level_level.csv csv file


In [6]:
if set(config.spetral_term).issubset(cal_configuration_set):
    logger.info(f"cal_loop {config.cal_loop_num} 组态耦合正确")
    cal_result = True
else:
    logger.error(f"cal_loop {config.cal_loop_num} 组态耦合错误")
    cal_result = False

2025-06-02 21:46:59,044 - INFO - cal_loop 1 组态耦合正确


In [10]:
rmix_file_load = gdp.GraspFileLoad.from_filepath(f'{cal_path}/{config.conf}_{config.cal_loop_num}.m', 'mix')
rmix_file_data = rmix_file_load.data_file_process()

Data file /home/workstation3/caldata/GdI/cvodd1/as3_odd4/cv4odd1as3_odd4_1/cv4odd1as3_odd4_1.m loaded.
data file type: rmcdhf mix_coefficient_data
g92mix: G92MIX
 nblock = 1,       ncftot =   385600,          nw =  41,            nelec =   64


100%|██████████| 1/1 [00:00<00:00, 445.26it/s]

cycle jblock = 1
 Block no. = 1, 2J+1 = 9, Parity = -1, No. of eigenvalues = 2, No. of CSFs = 385600

    Energy levels for ...
Rydberg constant is  109737.31568508

---------------------------------------------
 No Pos  J  Parity   Energy Total    Levels
                      (a.u.)         (cm^-1)
---------------------------------------------

  1  1   4   +    -11274.7413433        0.00
  2  0   4   +    -11274.7204780     4579.40


In [11]:
raw_csf_file_load = gdp.GraspFileLoad.from_filepath(f'{config.target_pool_file}', 'CSFs')
raw_csf_data = raw_csf_file_load.data_file_process()
    
indices_temp = [i for i in range(raw_csf_data.CSFs_block_length[0])]
sum_num_list = []
sum_num_min = round(math.ceil(raw_csf_data.CSFs_block_length[0] * config.initial_ratio))

Data file cv4odd1as3_odd4.c loaded.


In [12]:
for level in rmix_file_data.level_List:
    logger.info(f"             迭代能量：{level}")
logger.info(f"             耦合正确")
logger.info("************************************************")


2025-06-02 21:47:38,587 - INFO -              迭代能量：-11274.741343317686
2025-06-02 21:47:38,589 - INFO -              迭代能量：-11274.720478024547
2025-06-02 21:47:38,589 - INFO -              耦合正确
2025-06-02 21:47:38,589 - INFO - ************************************************


In [15]:
cut_off_csfs_indices_dict = gdp.batch_asfs_mix_square_above_threshold(rmix_file_data, float(config.cutoff_value))## !TODO config.cutoff_value需要在读取时就转为float类型


In [16]:
for index, value in cut_off_csfs_indices_dict.items():
    print(f"Index: {index}, {len(value)}")
    print(type(index))
    print(type(value))

Index: 0, 26971
<class 'int'>
<class 'numpy.ndarray'>


In [17]:
unique_indices = cut_off_csfs_indices_dict[0]
ci_desc = np.zeros(rmix_file_data.mix_coefficient_List[0].shape[1], dtype=bool)
ci_desc[unique_indices] = True

In [10]:
f'{cal_path}\\{config.conf}_{config.cal_loop_num}.c'



'D:\\PythonPrograms\\as3_odd4\\cv4odd1as3_odd4_1\\cv4odd1as3_odd4_1.c'

In [5]:
tgest_load = gdp.GraspFileLoad.from_filepath(f'{cal_path}\\{config.conf}_{config.cal_loop_num}.c', 'csf')
tgest_data= tgest_load.data_file_process()

Data file D:\PythonPrograms\as3_odd4\cv4odd1as3_odd4_1\cv4odd1as3_odd4_1.c loaded.


In [21]:
rmix_file_data.block_CSFs_nums

[np.int32(385600)]

In [6]:
t2 = tgest_data.CSFs_block_data[0][0][-1].rstrip()[5:-4].ljust(len(tgest_data.CSFs_block_data[0][0][0].rstrip()))

In [5]:
t1 = tgest_data.CSFs_block_data[0][0][-1].rstrip()[:-4].ljust(len(tgest_data.CSFs_block_data[0][0][0].rstrip()))


In [11]:
test_csf = ['  5s ( 2)  4d-( 4)  4d ( 6)  5p-( 2)  5p ( 4)  6s ( 2)  4f-( 3)  4f ( 4)  5d-( 1)',
            '                                                            5/2   2;   4      3/2',
            '                                                                        7/2      4-']

In [9]:
csfs_np = np.zeros( ( tgest_data.CSFs_block_length[0], 3 * len(asdg)), dtype=np.float32)

In [10]:
csfs_np.shape

(385600, 87)

In [ ]:
def subshell_charged_max_num(subshell_name: str, subshell_charged_num: int) -> bool:

    full_charged = {
        "s ": 2,
        "p-": 2,
        "p ": 4,
        "d-": 4,
        "d ": 6,
        "f-": 6,
        "f ": 8,
        "g-": 8,
        "g ": 10,
    }
    return full_charged.get(subshell_name, 0) == subshell_charged_num

In [ ]:
from typing import List, Tuple, Any, Optional

def parse_csf_2_descriptior(peel_subshells_List: List[str], csf: List[str], doubleJ: int) -> List[int]:
    ### 第二行分裂后，如果是全是空格，则为np.nan，如果有';'分号隔开，则选取最后一个
    
    subshells_line = csf[0].rstrip()
    line_length = len(subshells_line)
    middle_line = csf[1].rstrip().ljust(line_length)
    coupling_line = csf[2].rstrip()[:-4].ljust(line_length)
    
    subshell_List = gdp.chunk_string(subshells_line, 9)
    middle_line_List = gdp.chunk_string(middle_line, 9)
    coupling_line_List = gdp.chunk_string(coupling_line, 9)
    
    subshells_List_length = len(subshell_List)
    
    prev_j2cpl = 0
    
    for i, subshell_charged in enumerate(subshell_List):
        subshell = subshell_charged[:5]
        
        if not subshell:
            continue
        try:
            subshell_index = peel_subshells_List.index(subshell)
        except ValueError:
            print(f"Error: Subshell '{subshell}' not found in peel_subshells_List.")
            return []
        
        subshell_electron_num = int(subshell_charged[6:8])
        
        temp_middle_line_item = middle_line_List[i].split(';')
        
        if temp_middle_line_item[0].isspace():
            
        

In [18]:
dddf = '                                                            3/2   4;   4      3/2'

In [23]:
gdp.chunk_string(dddf.rstrip(), 9)

['         ',
 '         ',
 '         ',
 '         ',
 '         ',
 '         ',
 '      3/2',
 '   4;   4',
 '      3/2']

In [24]:
gdp.chunk_string(dddf.rstrip(), 9)[-3].split(';')

['      3/2']

In [ ]:
def generate_basis_pool_npy(npy_file_name, csfs_file_data: gdp.CSFs):
    peel_subshells_List = get_csfs_file_peel_subshells(csfs_file_data)
    
    csfs_np = np.zeros( ( tgest_data.CSFs_block_length[0], 3 * len(asdg)), dtype=np.float32)

In [ ]:
xzcvxcv = gdp.parse_csf_lines(asdg, test_csf, 8)

In [10]:
csftttt = tgest_data.CSFs_block_data[0][0]

In [8]:
asdg = gdp.get_CSFs_peel_subshells(tgest_data)

In [18]:
import re
from typing import Dict, Tuple, List

def process_csf_to_array(orb: List[str], csf: List[str]) -> np.ndarray:
    """
    处理CSF数据并生成numpy数组
    
    参数:
        orb: 轨道列表
        csf: CSF数据列表，包含三行信息
        
    返回:
        numpy数组，长度为orb长度的3倍
    """
    # 创建全零数组
    result = np.zeros(3 * len(orb), dtype=np.float32)
    
    # 处理CSF数据
    subshells_charges = csf[0].rstrip()
    middle_line = csf[1].rstrip().ljust(len(subshells_charges))
    coupling_line = csf[2].rstrip().ljust(len(subshells_charges))
    
    # 将CSF数据分割成块
    num_chunks = gdp.chunk_string(subshells_charges, 9)
    J2nu_chunks = gdp.chunk_string(middle_line, 9)
    J2cpl_chunks = gdp.chunk_string(coupling_line, 9)
    
    # 遍历每个轨道
    for i, current_orb in enumerate(orb):
        # 在subshells_charges中查找当前轨道
        for j, chunk in enumerate(num_chunks):
            if current_orb in chunk:
                # 提取电子数
                num_match = re.search(r'\((\d+)\)', chunk)
                if num_match:
                    num = int(num_match.group(1))
                    result[3*i] = num
                    
                    # 检查是否填满
                    subshell_name = current_orb[-2:]
                    if not gdp.if_subshell_full_charged(subshell_name, num):
                        # 处理耦合信息
                        J2nu_str = J2nu_chunks[j].strip()
                        if J2nu_str:
                            # 处理可能的多个耦合值
                            J2nu_parts = J2nu_str.split(';')
                            J2nu_str = J2nu_parts[-1].strip() if len(J2nu_parts) > 1 else J2nu_str
                            result[3*i+1] = gdp.J_to_doubleJ(J2nu_str)
                            
                            # 处理总耦合信息
                            J2cpl_str = J2cpl_chunks[j].strip()
                            result[3*i+2] = gdp.J_to_doubleJ(J2cpl_str) if J2cpl_str else result[3*i+1]
                break
    
    return result



In [20]:
import numpy as np
import pandas as pd
from tqdm import tqdm

def batch_convert_and_save_csf(CSFs_block_data, orb, save_path, file_format='npy'):
    """
    批量转化 CSFs_block_data 中的所有 CSF 数据并保存
    
    Args:
        CSFs_block_data: CSF块数据，格式为 List[blocks[CSF_items]]
        orb: 轨道列表
        save_path: 保存路径（不含扩展名）
        file_format: 保存格式 ('npy' 或 'csv')
    """
    all_csf_arrays = []
    
    # 遍历所有 block
    for block_idx, block in enumerate(tqdm(CSFs_block_data, desc="Processing blocks")):
        # 遍历每个 block 中的 CSF_item
        for csf_item in block:
            if len(csf_item) == 3:  # 确保是三行数据
                try:
                    arr = gdp.parse_csf_lines(orb, csf_item, 8)
                    all_csf_arrays.append(arr)
                except Exception as e:
                    print(f"Error processing CSF in block {block_idx}: {e}")
                    continue
    
    if not all_csf_arrays:
        print("No valid CSF data processed!")
        return
    
    # 转换为 numpy 数组
    all_csf_arrays = np.stack(all_csf_arrays)
    print(f"Total processed CSFs: {len(all_csf_arrays)}")
    print(f"Array shape: {all_csf_arrays.shape}")
    
    # 保存数据
    if file_format.lower() == 'npy':
        np.save(save_path + '.npy', all_csf_arrays)
        print(f"Data saved to: {save_path}.npy")
    elif file_format.lower() == 'csv':
        df = pd.DataFrame(all_csf_arrays)
        df.to_csv(save_path + '.csv', index=False)
        print(f"Data saved to: {save_path}.csv")
    else:
        raise ValueError("file_format must be 'npy' or 'csv'")

def batch_convert_and_save_csf_with_labels(CSFs_block_data, orb, save_path, file_format='csv'):
    """
    批量转化并保存，同时添加标签列（如果需要机器学习）
    
    Args:
        CSFs_block_data: CSF块数据
        orb: 轨道列表
        save_path: 保存路径
        file_format: 保存格式
    """
    all_csf_arrays = []
    labels = []  # 可以用来标记来源 block 或其他信息
    
    for block_idx, block in enumerate(tqdm(CSFs_block_data, desc="Processing blocks")):
        for csf_item in block:
            if len(csf_item) == 3:
                try:
                    arr = gdp.parse_csf_lines(orb, csf_item, 8)
                    all_csf_arrays.append(arr)
                    labels.append(block_idx)  # 记录来源 block
                except Exception as e:
                    print(f"Error processing CSF in block {block_idx}: {e}")
                    continue
    
    if not all_csf_arrays:
        print("No valid CSF data processed!")
        return
    
    all_csf_arrays = np.stack(all_csf_arrays)
    
    if file_format.lower() == 'csv':
        # 添加标签列
        df = pd.DataFrame(all_csf_arrays)
        df['block_label'] = labels
        df.to_csv(save_path + '.csv', index=False)
        print(f"Data with labels saved to: {save_path}.csv")
    else:
        # npy 格式单独保存标签
        np.save(save_path + '_data.npy', all_csf_arrays)
        np.save(save_path + '_labels.npy', np.array(labels))
        print(f"Data saved to: {save_path}_data.npy")
        print(f"Labels saved to: {save_path}_labels.npy")

In [19]:
# 基本用法
# asdg = get_CSFs_peel_subshells(tgest_data)
batch_convert_and_save_csf(
    tgest_data.CSFs_block_data,
    asdg,
    'test1',
    file_format='csv'
)

# 带标签的用法（适合机器学习）
batch_convert_and_save_csf_with_labels(
    tgest_data.CSFs_block_data,
    asdg,
    'test1_labels',
    file_format='csv'
)

Processing blocks: 100%|██████████| 1/1 [00:08<00:00,  8.40s/it]


Total processed CSFs: 385600
Array shape: (385600, 87)
Data saved to: test1.csv


Processing blocks: 100%|██████████| 1/1 [00:08<00:00,  8.38s/it]


Data with labels saved to: test1_labels.csv


In [21]:
# 基本用法
# asdg = get_CSFs_peel_subshells(tgest_data)
batch_convert_and_save_csf(
    tgest_data.CSFs_block_data,
    asdg,
    'test2',
    file_format='csv'
)

# 带标签的用法（适合机器学习）
batch_convert_and_save_csf_with_labels(
    tgest_data.CSFs_block_data,
    asdg,
    'test2_labels',
    file_format='csv'
)

Processing blocks: 100%|██████████| 1/1 [00:07<00:00,  7.13s/it]


Total processed CSFs: 385600
Array shape: (385600, 87)
Data saved to: test2.csv


Processing blocks: 100%|██████████| 1/1 [00:07<00:00,  7.39s/it]


Data with labels saved to: test2_labels.csv


In [13]:
xzcvxcv

[2,
 0,
 0,
 4,
 0,
 0,
 6,
 0,
 0,
 2,
 0,
 0,
 4,
 0,
 0,
 2,
 0,
 0,
 3,
 5,
 0,
 4,
 8,
 7,
 1,
 3,
 8,
 0,
 0,
 8,
 0,
 0,
 8,
 0,
 0,
 8,
 0,
 0,
 8,
 0,
 0,
 8,
 0,
 0,
 8,
 0,
 0,
 8,
 0,
 0,
 8,
 0,
 0,
 8,
 0,
 0,
 8,
 0,
 0,
 8,
 0,
 0,
 8,
 0,
 0,
 8,
 0,
 0,
 8,
 0,
 0,
 8,
 0,
 0,
 8,
 0,
 0,
 8,
 0,
 0,
 8,
 0,
 0,
 8,
 0,
 0,
 8]

In [16]:
tgest_data.CSFs_block_data[0][0]

['  5s ( 2)  4d-( 4)  4d ( 6)  5p-( 2)  5p ( 4)  6s ( 2)  4f-( 3)  4f ( 4)  5d-( 1)\n',
 '                                                            9/2        8      3/2\n',
 '                                                                        7/2      4-\n']

In [17]:
ddddd = pd.read_csv('/home/workstation3/caldata/GdI/cvodd1/as3_odd1/descripotors/cv4odd1as3_odd1_1_1_desc.csv')

In [18]:
ddddd

,0,1,2,3,4,5,6,7,8,9,...,78,79,80,81,82,83,84,85,86,0.1
0,2.0,0.0,0.0,4.0,0.0,0.0,6.0,0.0,0.0,2.0,...,0.0,0.0,3.0,0.0,0.0,3.0,0.0,0.0,3.0,True
1,2.0,0.0,0.0,4.0,0.0,0.0,6.0,0.0,0.0,2.0,...,0.0,0.0,3.0,0.0,0.0,3.0,0.0,0.0,3.0,True
2,2.0,0.0,0.0,4.0,0.0,0.0,6.0,0.0,0.0,2.0,...,0.0,0.0,3.0,0.0,0.0,3.0,0.0,0.0,3.0,True
3,2.0,0.0,0.0,4.0,0.0,0.0,6.0,0.0,0.0,2.0,...,0.0,0.0,3.0,0.0,0.0,3.0,0.0,0.0,3.0,True
4,2.0,0.0,0.0,4.0,0.0,0.0,6.0,0.0,0.0,2.0,...,0.0,0.0,3.0,0.0,0.0,3.0,0.0,0.0,3.0,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
180346,2.0,0.0,0.0,4.0,0.0,0.0,6.0,0.0,0.0,2.0,...,0.0,0.0,3.0,0.0,0.0,3.0,0.0,0.0,3.0,False
180347,2.0,0.0,0.0,4.0,0.0,0.0,6.0,0.0,0.0,2.0,...,0.0,0.0,3.0,0.0,0.0,3.0,0.0,0.0,3.0,False
180348,2.0,0.0,0.0,4.0,0.0,0.0,6.0,0.0,0.0,2.0,...,0.0,0.0,3.0,0.0,0.0,3.0,0.0,0.0,3.0,False
180349,2.0,0.0,0.0,4.0,0.0,0.0,6.0,0.0,0.0,2.0,...,0.0,0.0,3.0,0.0,0.0,3.0,0.0,0.0,3.0,False


In [19]:
X = ddddd.iloc[:, :-1]

In [20]:
X

,0,1,2,3,4,5,6,7,8,9,...,77,78,79,80,81,82,83,84,85,86
0,2.0,0.0,0.0,4.0,0.0,0.0,6.0,0.0,0.0,2.0,...,3.0,0.0,0.0,3.0,0.0,0.0,3.0,0.0,0.0,3.0
1,2.0,0.0,0.0,4.0,0.0,0.0,6.0,0.0,0.0,2.0,...,3.0,0.0,0.0,3.0,0.0,0.0,3.0,0.0,0.0,3.0
2,2.0,0.0,0.0,4.0,0.0,0.0,6.0,0.0,0.0,2.0,...,3.0,0.0,0.0,3.0,0.0,0.0,3.0,0.0,0.0,3.0
3,2.0,0.0,0.0,4.0,0.0,0.0,6.0,0.0,0.0,2.0,...,3.0,0.0,0.0,3.0,0.0,0.0,3.0,0.0,0.0,3.0
4,2.0,0.0,0.0,4.0,0.0,0.0,6.0,0.0,0.0,2.0,...,3.0,0.0,0.0,3.0,0.0,0.0,3.0,0.0,0.0,3.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
180346,2.0,0.0,0.0,4.0,0.0,0.0,6.0,0.0,0.0,2.0,...,3.0,0.0,0.0,3.0,0.0,0.0,3.0,0.0,0.0,3.0
180347,2.0,0.0,0.0,4.0,0.0,0.0,6.0,0.0,0.0,2.0,...,3.0,0.0,0.0,3.0,0.0,0.0,3.0,0.0,0.0,3.0
180348,2.0,0.0,0.0,4.0,0.0,0.0,6.0,0.0,0.0,2.0,...,3.0,0.0,0.0,3.0,0.0,0.0,3.0,0.0,0.0,3.0
180349,2.0,0.0,0.0,4.0,0.0,0.0,6.0,0.0,0.0,2.0,...,3.0,0.0,0.0,3.0,0.0,0.0,3.0,0.0,0.0,3.0


In [21]:
y = ddddd.iloc[:, -1]

In [22]:
y

0          True
1          True
2          True
3          True
4          True
          ...  
180346    False
180347    False
180348    False
180349    False
180350    False
Name: 0.1, Length: 180351, dtype: bool

In [23]:
from sklearn.model_selection import train_test_split

In [24]:
X_train, X_test, y_train, y_test = train_test_split(X.values, y.values, test_size=0.2, random_state=42)

In [28]:
X_train.shape

(144280, 87)

In [27]:
X_test.shape

(36071, 87)